# Baomai: test the reviewed training boxes

**Select a GPU runtime, run all cells, then download and return one ZIP.** This notebook verifies a fresh checkout and the original inputs, runs a short resource trial, and trains a matched reference/candidate pair in one session.

Both arms start freshly from **epoch173** and use the same `box-supervision-warmup-v1` recipe: 30 epochs, batch 4, image size 1280, FP32, AdamW, seed 42 and `warmup_bias_lr=0.0001`. The scientific change is limited to **16 accepted box-edge corrections** in three training images: 8 in `new_000030`, 6 in `new_000012` and 2 in `new_000128`. The candidate preserves every object row, class and endpoint; validation remains unchanged. Unresolved annotation fields stay at their original values, so this is a bounded correction experiment.

| Run | Training labels | Recipe | Initialization |
| --- | --- | --- | --- |
| Reference | Original frozen labels | `box-supervision-warmup-v1` | Original epoch173 |
| Candidate | Reviewed box overlay | Same | Original epoch173 |

Keep epoch173 as the working model while this experiment is assessed. Training completion or the trainer's `best.pt` score does not establish improved accuracy.

Training source is pinned to [`SOURCE_COMMIT_TO_PIN`](https://github.com/Siuuuers/dl/tree/SOURCE_COMMIT_TO_PIN). The [candidate manifest](https://github.com/Siuuuers/dl/blob/SOURCE_COMMIT_TO_PIN/data/candidates/box-refinement-2026-11-07/manifest.json) records the bounded edits under the [annotation policy](https://github.com/Siuuuers/dl/blob/SOURCE_COMMIT_TO_PIN/docs/annotation-policy.md). This handoff contains no precomputed outputs; no GPU run of this new experiment has been performed here.

## 1. Check the GPU and create a fresh workspace

In Colab, select **Runtime → Change runtime type → GPU**, then choose **Runtime → Run all**. Leave the session unchanged until both runs and packaging finish.

The notebook creates `/content/baomai-box-supervision-20261107`. An existing directory is refused so previous results remain available. For a new attempt in the same runtime, choose an unused `RUN_BASE` path below. Save completed results before resetting or closing the runtime.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import subprocess
import sys
import time

SOURCE_REVISION = "SOURCE_COMMIT_TO_PIN"
RECIPE_ID = "box-supervision-warmup-v1"
CANDIDATE_MANIFEST = "data/candidates/box-refinement-2026-11-07/manifest.json"
RUN_BASE = Path("/content/baomai-box-supervision-20261107")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the handoff notebook with its full source commit pinned."
assert sys.version_info >= (3, 11), "Python 3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run([sys.executable, "-c", "import torch, torchvision; "
     "assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'; "
     "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda); "
     "print('GPU:', torch.cuda.get_device_name(0)); "
     "print('CUDA calculation:', (torch.ones(1, device='cuda:0') * 2).item())"])
run(["nvidia-smi"])
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    raise FileExistsError(f"Workspace already exists: {RUN_BASE}. Preserve its results and choose a new RUN_BASE.")
RUN_BASE.mkdir(parents=True, exist_ok=False)
REPO = RUN_BASE / "dl"
print("Workspace:", RUN_BASE)


## 2. Download and verify the pinned code and original inputs

Git supplies code, labels and the reviewed box overlay. Selective Git LFS downloads only the **157 original training/validation PNGs and epoch173 weights**. The file inventory is checked against the frozen manifest before downloading.

The read-only preflight then verifies all original PNG hashes and dimensions, both label sets, the candidate's exact edit inventory and the original checkpoint bytes. No model inference is run. You do not need to upload an earlier input ZIP.

In [ ]:
lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
run(["git", "init", REPO], env=git_env)
run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
run(["git", "fetch", "--depth=1", "origin", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, env=git_env)
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"],
    cwd=REPO, text=True).splitlines())
assert len(reference_manifest["records"]) == len(expected_images) == 157
assert tracked_images == expected_images, "The LFS image selection differs from the frozen manifest."
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt",
     "--exclude="], cwd=REPO)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REVISION
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()

input_preflight = """
import json
from pathlib import Path
import sys
sys.dont_write_bytecode = True
root = Path.cwd()
sys.path.insert(0, str(root / 'scripts'))
from evaluate_pose import preflight_dataset
from stage_pose_training import source_state
from pose_checkpoint import get_recipe, recipe_identity
candidate = root / sys.argv[1]
recipe_id = sys.argv[2]
proposal = json.loads(candidate.read_text())
assert proposal['kind'] == 'training_box_overlay'
assert proposal['candidate_id'] == 'box-refinement-2026-11-07'
edge_counts = {change['image_id']: sum(len(row['box_edges_changed']) for row in change['rows'])
               for change in proposal['changes']}
assert edge_counts == {'new_000030': 8, 'new_000012': 6, 'new_000128': 2}, edge_counts
state = source_state(root, candidate)
initial = preflight_dataset(root, require_weights=True)
assert len(state['records']) == 157
assert state['counts']['reference'] == state['counts']['candidate']
assert state['counts']['reference']['train']['images'] == 137
assert state['counts']['reference']['val']['images'] == 20
assert state['source']['initial_checkpoint_sha256'] == initial['model_sha256']
assert Path(initial['weights']) == root / 'weights/cassette_lower_epoch173.pt'
assert get_recipe(recipe_id) == get_recipe('warmup-bias-lr-1e-4-v1')
print(json.dumps({'status': 'verified', 'original_images': len(state['records']),
                  'box_edge_corrections': edge_counts, 'label_counts': state['counts'],
                  'initial_checkpoint_sha256': initial['model_sha256'],
                  'weights_verified': initial['weights_verified'],
                  **recipe_identity(recipe_id)}, indent=2))
"""
run([sys.executable, "-c", input_preflight, CANDIDATE_MANIFEST, RECIPE_ID], cwd=REPO)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified clean source:", actual_revision)


## 3. Preserve CUDA PyTorch and install the pinned trainer

The installed `torch` and `torchvision` versions are constrained during installation. If this cell reports a dependency conflict, resolve it before training; do not discard the constraints and silently change the GPU runtime. Training uses fresh subprocesses, so it sees the installed packages without importing the trainer into this notebook's kernel.

In [ ]:
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text(
    ''.join(f'{name}=={version(name)}\n' for name in ('torch', 'torchvision')), encoding='utf-8'
)
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; assert torch.cuda.is_available(); "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__, torch.cuda.get_device_name(0))"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or select a larger-memory runtime, then rerun this cell."
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Time budgets: 600 seconds for the trial; 21600 seconds per complete run.")


## 4. Run a short real training trial

The trial exercises the candidate labels with the same full 30-epoch schedule used by both later runs, then stops after **four batches**. It must report actual optimizer updates and an observed parameter change before the notebook continues.

Its time budget is 600 seconds. Each complete run has a six-hour stop budget; these limits are not duration estimates. The printed memory limit covers **host RAM**, while GPU VRAM is shown by `nvidia-smi`. This trial does not qualify the later validation or checkpoint-finalization memory costs and does not produce an eligible model.

Progress appears below every 20 seconds; detailed trainer output is retained in each run's `train.log`.

In [ ]:
def train_one(folder, arm, *, trial=False):
    assert arm in ("reference", "candidate")
    output = RUN_BASE / folder
    command = [sys.executable, "scripts/train_pose_comparison.py",
               "--trial" if trial else "--train", "--arm", arm, "--recipe", RECIPE_ID,
               "--candidate", CANDIDATE_MANIFEST,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, arm, RECIPE_ID, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}

    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass

    started = time.monotonic()
    last_notice = -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; status={progress.get('status')}; "
                          f"completed epochs={len(progress.get('completed_epochs', []))}/30; "
                          f"batches={progress.get('batches_completed', 0)}; "
                          f"updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: {elapsed:.0f}s; verifying/staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            receipt_file = output / "training-receipt.json"
            if receipt_file.exists():
                failed_receipt = json.loads(receipt_file.read_text())
                for key in ("error", "input_verification_error", "checkpoint_manifest_error"):
                    if failed_receipt.get(key):
                        print(f"{key}: {failed_receipt[key]}", flush=True)
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed. Preserve its receipt/log and use a new output directory for a retry.")
    except BaseException:
        # Stop this invocation's descendants if the notebook cell is interrupted.
        # Partial runs remain ineligible; no receipt is fabricated here.
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = json.loads((output / "training-receipt.json").read_text())
    assert receipt["status"] == ("trial-complete" if trial else "complete")
    assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
    assert receipt["arm"] == arm
    assert receipt["recipe_id"] == RECIPE_ID
    assert receipt["initial_checkpoint_sha256"] == reference_manifest["model_sha256"]
    assert receipt["training_settings"]["epochs"] == 30
    assert receipt["training_settings"]["warmup_bias_lr"] == 0.0001
    if not trial:
        assert receipt["completed_epochs"] == list(range(1, 31))
    assert (output / "checkpoint-manifest.json").exists() is (not trial)
    return output

TRIAL = train_one("resource-trial", "candidate", trial=True)
print("Resource trial passed. This is not a completed model.")


## 5. Train the reference and candidate sequentially

The original-label reference runs first, then the reviewed-box candidate. **Both use the same low-bias recipe and start freshly from epoch173.** Neither starts from the trial or from the other run. Keep the source, GPU session and environment unchanged through both runs.

A failed or interrupted run is ineligible for packaging. Preserve its receipt and log, diagnose the failure, and retry in a fresh output directory. If only the candidate needs retrying in this otherwise unchanged session, use `CANDIDATE = train_one("candidate-retry-001", "candidate")`, then continue to packaging. If code, environment or scientific settings changed, start a new matched pair.

In [ ]:
REFERENCE = train_one("reference", "reference")
CANDIDATE = train_one("candidate", "candidate")
for arm, folder in (("reference", REFERENCE), ("candidate", CANDIDATE)):
    receipt = json.loads((folder / "training-receipt.json").read_text())
    print(arm, "recipe:", receipt["recipe_id"], "epochs:", len(receipt["completed_epochs"]),
          "batches:", receipt["batches_completed"], "optimizer updates:", receipt["optimizer_steps"],
          "GPU:", receipt["runtime"]["device_name"])


## 6. Verify and package the matched results

The existing paired-results packer checks complete run eligibility, source/runtime identity, the dataset proposal and shared training settings. It includes each finalized epoch-30 `last.pt` together with its receipt, checkpoint manifest, staging manifest, CSV results and raw log. Failed runs and mismatched pairs are refused.

Packaging records the experiment; it does not approve a model or alter any evaluation or inference configuration. Original images and intermediate checkpoints do not need to be returned.

In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-box-supervision-results.zip"
run([sys.executable, "scripts/package_gpu_results.py", "--reference", REFERENCE,
     "--candidate", CANDIDATE, "--output", RESULTS_ZIP], cwd=REPO)
with RESULTS_ZIP.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()
print("Return:", RESULTS_ZIP.name)
print("Bytes:", RESULTS_ZIP.stat().st_size)
print("SHA-256:", digest)


## 7. Download and return the ZIP

Attach **`baomai-box-supervision-results.zip`** in chat or return it through a new result PR before closing the runtime. If the browser blocks or interrupts the transfer, run this download cell again. It reuses the verified archive and does not repeat training or packaging.

In [ ]:
assert RESULTS_ZIP.is_file(), "Finish the packaging cell first."
from google.colab import files
files.download(str(RESULTS_ZIP))


### What happens after the return

Evaluate epoch173, the fresh reference and the box-supervision candidate together under the same frozen validation images, scorer and thresholds. Use each returned checkpoint manifest to select the predeclared final-epoch model. Assess the candidate against both epoch173 and its matched reference before changing the working model.

This notebook performs the training handoff and packaging only. It does not automatically run the final comparison, tune thresholds or change the default model. The one-seed experiment on an already-used development-validation set can test the effect of these bounded box corrections; it cannot establish independent-test accuracy or resolve the remaining annotation ambiguities.